# Function Limits and Numerical Approximations

This notebook explores the fundamental mathematical concept of a **limit**, the foundational pillar upon which derivatives, integrals, and gradient-based optimization are built.

## Mathematical Formulation

The limit of a function $f(x)$ as $x$ approaches a point $c$ is written as:

$$\lim_{x \to c} f(x) = L$$

Informally, this means that as $x$ gets arbitrarily close to $c$ (from either side, without necessarily equaling $c$), the value of $f(x)$ gets arbitrarily close to $L$.

A two-sided limit exists if and only if both one-sided limits exist and are equal:

$$\lim_{x \to c^-} f(x) = \lim_{x \to c^+} f(x) = L$$

In this notebook, we examine:
1. Array generation with `np.arange` and numerical evaluation.
2. Fundamental limits: $\lim_{x \to 0} \frac{\sin x}{x} = 1$.
3. Removable discontinuities: $f(x) = \frac{x^2 - 4}{x - 2}$.
4. Jump discontinuities: $f(x) = \frac{|x - 2|}{x - 2}$.
5. Infinite discontinuities (asymptotes): $f(x) = \frac{1}{(x - 2)^2}$ and $f(x) = \frac{1}{x}$.
6. Piecewise functions and directional continuity.
7. Asymptotic limits: Euler's number $e$ and exponential growth rates.


In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

# Set clean aesthetic defaults for matplotlib
plt.rcParams["figure.figsize"] = [7.0, 4.5]
plt.rcParams["figure.autolayout"] = True

## 1. Sequence Generation with `np.arange`

Before evaluating limits numerically, we construct discretized evaluation grids using `np.arange(start, stop, step)`. Notice how floating-point steps can introduce minor precision offsets, which can be inspected directly.


In [ ]:
# Discretized domain grid
data = np.arange(0, 10, 0.5).tolist()
print("Discretized grid sample (first 8 values):", data[:8])
print("Grid length:", len(data))
assert len(data) == 20

## 2. Fundamental Limit: $\lim_{x \to 0} \frac{\sin(x)}{x} = 1$

At $x = 0$, direct evaluation yields the indeterminate form $\frac{0}{0}$. By approaching $0$ from both the positive and negative directions, we observe convergence toward $1.0$.


In [ ]:
def sinc_func(x: float) -> float:
    """Compute sin(x) / x for x != 0."""
    return math.sin(x) / x


# Approach from right (t -> 0+)
t = 1.0
print("Approaching 0 from the right (t -> 0+):")
for _step in range(7):
    y = sinc_func(t)
    print(f"  t = {t:10.7f} -> sinc(t) = {y:12.9f}")
    t *= 0.1

# Approach from left (t -> 0-)
t = 1.0
print("\nApproaching 0 from the left (t -> 0-):")
for _step in range(7):
    y = sinc_func(-t)
    print(f"  t = {-t:10.7f} -> sinc(-t) = {y:12.9f}")
    t *= 0.1

In [ ]:
# Visualizing sinc_func around the origin
left_x = np.arange(-6.0, -1e-6, 0.02).tolist()
left_y = [sinc_func(x) for x in left_x]

right_x = np.arange(1e-6, 6.0, 0.02).tolist()
right_y = [sinc_func(x) for x in right_x]

plt.figure()
plt.plot(left_x, left_y, "b-", label=r"$x < 0$")
plt.plot(right_x, right_y, "g-", label=r"$x > 0$")
plt.plot([0], [1.0], "ro", markerfacecolor="white", markersize=7, label=r"Limit point $(0, 1)$")
plt.axhline(1.0, color="gray", linestyle="--", alpha=0.6)
plt.axvline(0.0, color="gray", linestyle="--", alpha=0.6)
plt.title(r"Limit of $\frac{\sin(x)}{x}$ as $x \to 0$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Removable Discontinuity: $f(x) = \frac{x^2 - 4}{x - 2}$

Notice that:
$$\frac{x^2 - 4}{x - 2} = \frac{(x - 2)(x + 2)}{x - 2} = x + 2 \quad \text{for } x \neq 2$$

Thus, although $f(2)$ is undefined, $\lim_{x \to 2} f(x) = 2 + 2 = 4$.


In [ ]:
def rational_removable(x: float) -> float:
    """Compute (x^2 - 4) / (x - 2)."""
    return (x * x - 4.0) / (x - 2.0)


eps = 1.0
print("Approaching x=2 from right and left:")
for _step in range(7):
    y_right = rational_removable(2.0 + eps)
    y_left = rational_removable(2.0 - eps)
    print(f"  eps = {eps:10.7f} | f(2 + eps) = {y_right:10.7f} | f(2 - eps) = {y_left:10.7f}")
    eps *= 0.1

# Verify limit value numerically
assert abs(rational_removable(2.0 + 1e-7) - 4.0) < 1e-6

In [ ]:
# Visualization of removable discontinuity
left_x = np.arange(0.0, 1.99, 0.02).tolist()
left_y = [rational_removable(x) for x in left_x]

right_x = np.arange(2.01, 4.0, 0.02).tolist()
right_y = [rational_removable(x) for x in right_x]

plt.figure()
plt.plot(left_x, left_y, "g-", label=r"$f(x), x < 2$")
plt.plot(right_x, right_y, "b-", label=r"$f(x), x > 2$")
plt.plot([2], [4], "ro", markerfacecolor="white", markersize=8, label="Removable hole (2, 4)")
plt.title(r"Removable Discontinuity: $f(x) = \frac{x^2 - 4}{x - 2}$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 4. Jump Discontinuity: $f(x) = \frac{|x - 2|}{x - 2}$

Here, the left-hand limit and right-hand limit do not coincide:
$$\lim_{x \to 2^-} \frac{|x - 2|}{x - 2} = -1, \qquad \lim_{x \to 2^+} \frac{|x - 2|}{x - 2} = +1$$

Because $\lim_{x \to 2^-} f(x) \neq \lim_{x \to 2^+} f(x)$, the two-sided limit **does not exist**.


In [ ]:
def sign_discontinuity(x: float) -> float:
    return abs(x - 2.0) / (x - 2.0)


t = 1.0
print("Testing jump discontinuity at x = 2:")
for _step in range(5):
    y_right = sign_discontinuity(2.0 + t)
    y_left = sign_discontinuity(2.0 - t)
    print(f"  t = {t:8.5f} | f(2 + t) = {y_right:+4.1f} | f(2 - t) = {y_left:+4.1f}")
    t *= 0.1

assert sign_discontinuity(2.0 + 1e-4) == 1.0
assert sign_discontinuity(2.0 - 1e-4) == -1.0

In [ ]:
left_x = np.arange(0.0, 1.99, 0.02).tolist()
left_y = [sign_discontinuity(x) for x in left_x]

right_x = np.arange(2.01, 4.0, 0.02).tolist()
right_y = [sign_discontinuity(x) for x in right_x]

plt.figure()
plt.plot(left_x, left_y, "g-", lw=2, label=r"Left branch ($y = -1$)")
plt.plot(right_x, right_y, "b-", lw=2, label=r"Right branch ($y = +1$)")
plt.plot([2], [-1], "go", markerfacecolor="white", markersize=7)
plt.plot([2], [1], "bo", markerfacecolor="white", markersize=7)
plt.title(r"Jump Discontinuity: $f(x) = \frac{|x - 2|}{x - 2}$")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.ylim(-2, 2)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. Infinite Limits (Vertical Asymptotes)

Consider $f(x) = \frac{1}{(x - 2)^2}$ and $g(x) = \frac{1}{x}$.

For $f(x)$, as $x \to 2$, both sides grow toward $+\infty$.
For $g(x)$, as $x \to 0$, $g(x) \to +\infty$ from the right and $-\infty$ from the left.


In [ ]:
def infinite_pole(x: float) -> float:
    return 1.0 / ((x - 2.0) ** 2)


t = 1.0
print("f(x) = 1 / (x - 2)^2 as x -> 2:")
for _step in range(6):
    print(
        f"  t = {t:8.5f} | f(2 + t) = {infinite_pole(2.0 + t):14.2f} | f(2 - t) = {infinite_pole(2.0 - t):14.2f}"
    )
    t *= 0.1

In [ ]:
def reciprocal(x: float) -> float:
    return 1.0 / x


t = 1.0
print("\ng(x) = 1 / x as x -> 0:")
for _step in range(6):
    print(f"  t = {t:8.5f} | g(+t) = {reciprocal(t):+14.2f} | g(-t) = {reciprocal(-t):+14.2f}")
    t *= 0.1

## 6. Piecewise Functions and Directional Continuity

Consider the piecewise function:
$$f(x) = \begin{cases} x + 1 & \text{if } x < 2 \\ x^2 - 4 & \text{if } x \ge 2 \end{cases}$$

Here, $\lim_{x \to 2^-} f(x) = 2 + 1 = 3$, whereas $\lim_{x \to 2^+} f(x) = 2^2 - 4 = 0$. Since $f(2) = 0$, the function is right-continuous but not continuous at $x = 2$.


In [ ]:
def piecewise_function(x: float) -> float:
    if x < 2.0:
        return x + 1.0
    return x * x - 4.0


t = 1.0
print("Piecewise limit test around x = 2:")
for _step in range(5):
    print(
        f"  t = {t:8.5f} | f(2 + t) = {piecewise_function(2.0 + t):8.5f} | f(2 - t) = {piecewise_function(2.0 - t):8.5f}"
    )
    t *= 0.1

In [ ]:
left_x = np.arange(0.0, 2.0, 0.02).tolist()
left_y = [piecewise_function(x) for x in left_x]

right_x = np.arange(2.0, 4.0, 0.02).tolist()
right_y = [piecewise_function(x) for x in right_x]

plt.figure()
plt.plot(left_x, left_y, "g-", lw=2, label=r"$x + 1$ for $x < 2$")
plt.plot(right_x, right_y, "b-", lw=2, label=r"$x^2 - 4$ for $x \ge 2$")
plt.plot([2], [3], "go", markerfacecolor="white", markersize=7, label=r"Open limit $f(2^-) = 3$")
plt.plot([2], [0], "bo", markerfacecolor="blue", markersize=7, label=r"Closed value $f(2) = 0$")
plt.title("Piecewise Function Discontinuity")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Fundamental Limit: Euler's Number $e$

Euler's number $e \approx 2.718281828...$ is defined as the asymptotic limit:

$$e = \lim_{x \to \infty} \left(1 + \frac{1}{x}\right)^x$$

Similarly, for any base $a > 0$, the instantaneous rate of exponential growth satisfies:

$$\lim_{t \to 0} \frac{a^t - 1}{t} = \ln(a)$$


In [ ]:
# Asymptotic limit for Euler's constant e
def e_limit(x: float) -> float:
    return (1.0 + 1.0 / x) ** x


print("Approximating Euler's number e:")
n = 10.0
for _step in range(8):
    val = e_limit(n)
    err = abs(val - math.e)
    print(f"  n = {n:12.1f} -> (1 + 1/n)^n = {val:14.9f} (error: {err:10.2e})")
    n *= 10.0

assert abs(e_limit(1e8) - math.e) < 1e-6

In [ ]:
# Derivative limit of general exponential base
def exp_rate_limit(a: float, t: float) -> float:
    return (a**t - 1.0) / t


print("Rate limit (a^t - 1) / t as t -> 0:")
for base in [2.0, math.e, 10.0]:
    t = 1.0
    for _ in range(6):
        t *= 0.1
    approx = exp_rate_limit(base, t)
    truth = math.log(base)
    print(
        f"  base a = {base:6.3f} | approx = {approx:10.6f} | ln(a) = {truth:10.6f} | diff = {abs(approx - truth):10.2e}"
    )

## Summary

1. Limits allow us to evaluate the local trend of a function even when direct substitution leads to indeterminate forms such as $\frac{0}{0}$ or $\frac{\infty}{\infty}$.
2. A two-sided limit exists if and only if both left and right directional limits converge to the same finite value.
3. In machine learning, numerical stability often requires carefully handling limit behaviors (e.g., in $\text{softmax}$, cross-entropy, and log-sum-exp operations) to prevent division by zero or numerical underflow.
